# 📊 Análise Socioeconômica — Dados IBGE

Análise exploratória de **76.840 registros** com informações de renda, escolaridade, cor/raça e localização da população brasileira.

**Variáveis disponíveis:** UF, Sexo, Idade, Cor, Anos de Estudo, Renda, Altura

## 1. Importações e Carregamento dos Dados

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

dados = pd.read_csv('dados.csv')
dados.head()

## 2. Mapeamento das Variáveis
Os dados originais usam códigos numéricos. Aqui converti para valores legíveis.

In [ ]:
dados['Sexo'] = dados['Sexo'].map({0: 'Masculino', 1: 'Feminino'})

cor = {0: 'Indígena', 2: 'Branca', 4: 'Preta', 6: 'Amarela', 8: 'Parda', 9: 'Sem declaração'}
dados['Cor'] = dados['Cor'].map(cor)

anos_de_estudo = {
    2: '1 ano', 3: '2 anos', 4: '3 anos', 5: '4 anos',
    6: '5 anos', 7: '6 anos', 8: '7 anos', 9: '8 anos',
    10: '9 anos', 11: '10 anos', 12: '11 anos', 13: '12 anos',
    14: '13 anos', 15: '14 anos', 16: '15 ou mais'
}
dados['Anos de Estudo'] = dados['Anos de Estudo'].map(anos_de_estudo)

uf = {
    11: 'Rondônia', 12: 'Acre', 13: 'Amazonas', 14: 'Roraima', 15: 'Pará',
    16: 'Amapá', 17: 'Tocantins', 21: 'Maranhão', 22: 'Piauí', 23: 'Ceará',
    24: 'Rio Grande do Norte', 25: 'Paraíba', 26: 'Pernambuco', 27: 'Alagoas',
    28: 'Sergipe', 29: 'Bahia', 31: 'Minas Gerais', 32: 'Espírito Santo',
    33: 'Rio de Janeiro', 35: 'São Paulo', 41: 'Paraná', 42: 'Santa Catarina',
    43: 'Rio Grande do Sul', 50: 'Mato Grosso do Sul', 51: 'Mato Grosso',
    52: 'Goiás', 53: 'Distrito Federal'
}
dados['UF'] = dados['UF'].map(uf)

print("Dados mapeados com sucesso!")
dados.head()

## 3. Análises Gerais

In [ ]:
rendaMediaT = dados['Renda'].mean()
idadeMediaT = dados['Idade'].mean()
alturaMedia = dados.groupby('Sexo')['Altura'].mean()
npcor = dados['Cor'].value_counts()
pessoasporUF = dados['UF'].value_counts()

print(f'Renda média geral: R$ {rendaMediaT:.2f}')
print(f'Idade média: {idadeMediaT:.1f} anos')
print(f'\nAltura média por sexo:\n{alturaMedia}')
print(f'\nPessoas por cor/raça:\n{npcor}')
print(f'\nPessoas por UF:\n{pessoasporUF}')

## 4. Análises de Gênero

In [ ]:
rendaMediaSexo = dados.groupby('Sexo')['Renda'].mean()
diferenca = rendaMediaSexo['Masculino'] - rendaMediaSexo['Feminino']
print(f'Renda média por sexo:\n{rendaMediaSexo}')
print(f'\nDiferença salarial: R$ {diferenca:.2f}')

In [ ]:
anosDeEstudoSexo = dados['Anos de Estudo'].groupby(dados['Sexo']).value_counts()
anosDeEstudoSexo = anosDeEstudoSexo.reindex(anos_de_estudo.values(), level='Anos de Estudo')
print(anosDeEstudoSexo)

In [ ]:
limites = [1.00, 1.60, 1.70, 1.80, 1.90, 2.20]
labels  = ['Abaixo de 1.60', 'Até 1.70', 'Até 1.80', 'Até 1.90', 'Acima de 1.90']
alturaPorSexo = pd.cut(x=dados.Altura, bins=limites, labels=labels, include_lowest=True)\
    .groupby(dados['Sexo']).value_counts()\
    .reindex(labels, level='Altura')
print(alturaPorSexo)

## 5. Análises de Renda

In [ ]:
limites = [0, 1576, 3152, 7880, 15760, 200000]
labels  = ['E', 'D', 'C', 'B', 'A']
classes = pd.cut(x=dados.Renda, bins=limites, labels=labels, include_lowest=True)\
    .value_counts().sort_index(ascending=False)
print('Distribuição por classe social:')
print(classes)

In [ ]:
rendaMediaUF = dados.groupby('UF')['Renda'].mean().sort_values(ascending=False)
estadoMaisRico  = rendaMediaUF.idxmax()
estadoMaisPobre = rendaMediaUF.idxmin()
print(f'Estado mais rico:  {estadoMaisRico} (R$ {rendaMediaUF[estadoMaisRico]:.2f})')
print(f'Estado mais pobre: {estadoMaisPobre} (R$ {rendaMediaUF[estadoMaisPobre]:.2f})')
print(f'\nRenda média por cor/raça:')
print(dados.groupby('Cor')['Renda'].mean().sort_values(ascending=False))

## 6. Análises de Educação

In [ ]:
anosDeEstudoCor = dados.groupby('Cor')['Anos de Estudo']\
    .value_counts()\
    .reindex(anos_de_estudo.values(), level='Anos de Estudo')
print(anosDeEstudoCor)

In [ ]:
anosDeEstudoUF = dados.groupby('UF')['Anos de Estudo']\
    .value_counts()\
    .reindex(anos_de_estudo.values(), level='Anos de Estudo')
print(anosDeEstudoUF)

## 7. Visualização — Renda Média por Anos de Estudo

Este gráfico mostra a relação entre escolaridade e renda. Quanto mais anos de estudo, maior a renda média.

In [ ]:
rendaMediaAnosDeEstudo = dados[dados['Renda'].between(1, 20000)]\
    .groupby('Anos de Estudo')['Renda']\
    .mean()\
    .reindex(anos_de_estudo.values())

ax = sns.lineplot(x=rendaMediaAnosDeEstudo.index, y=rendaMediaAnosDeEstudo.values)
ax.figure.set_size_inches(15, 6)
ax.set_title('Renda Média por Anos de Estudo', fontsize=14)
ax.set_xlabel('Anos de Estudo', fontsize=12)
ax.set_ylabel('Renda Média (R$)', fontsize=12)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 8. Conclusões

- A **renda média geral** da população é de R$ 2.000,38
- Homens ganham em média **28,5% a mais** que mulheres
- **64,8%** da população está na classe E (até R$ 1.576)
- O **Distrito Federal** é o estado com maior renda média e o **Maranhão** o mais pobre
- Existe uma relação positiva clara entre **anos de estudo e renda**
- A cor **Amarela** apresenta a maior renda média e a **Preta** a menor